# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Umair01-code/FlyRank-ML-Intrenship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

Lane: **Refresh / Content Opportunity Scoring.** Everything below uses the March 2026 warehouse slice from my Week 3 contract and Week 4 baseline. Random seed is 42 everywhere.

## 1. Method choice and why

My question for this notebook: **which pages should a content team look at first because they are slipping?**

To turn that into something I can model I need a label I can actually observe, so I build weekly rows from the daily warehouse data. One row is one page (client + content item) at one decision date.

- **Features:** only the 7 days before the decision date, plus the 7 days before that so the model can see momentum.
- **Label:** 1 if the page's clicks in the next 7 days are at least 30% lower than in the previous 7 days, otherwise 0.
- **Eligible pages:** at least 10 clicks in the previous week, and a complete 7-day Search Console history in both the "before" and "after" windows. Without this filter a page going from 1 click to 0 would count as a big drop, and a few missing days would look like a drop too.

I moved from my Week 3 regression label (`future_7d_gsc_clicks`) to this yes/no version on purpose. My lane is about ranking pages for review, and my Week 2 success metric was Precision@50, which needs a yes/no label. It uses the same two click columns, so nothing from the future leaks into the features.

**Why these methods (from this week's menu):**

- The label is observed and yes/no, so I start with **Logistic Regression** because I can read it, then a **depth-3 Decision Tree** I can print, then **Random Forest** and **Gradient Boosting** to see if the extra complexity earns anything.
- I am not using clustering. There is a label, so grouping pages without one does not answer the question.
- **Permutation importance** at the end, to check what the best model really leans on.
- The **Week 4 rule baseline** goes in the same table. It ranks pages with high visibility and weak CTR. It was not built to predict drops, so I do not expect it to do well here. That is a fair thing to find out: is "looks like an opportunity" the same as "is slipping"?

In [9]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from pathlib import Path

warnings.filterwarnings("ignore")
SEED = 42
rng = np.random.default_rng(SEED)

print("pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)

pandas 2.2.3 | numpy 2.4.6 | scikit-learn 1.6.1


In [10]:
# Warehouse connection. The token comes from Colab Secrets and is never printed.
%pip -q install duckdb

import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{userdata.get("HF_TOKEN")}'
    )
""")
print("Connected to the warehouse.")

Connected to the warehouse.


In [6]:
!pip install pandas duckdb pyarrow

In [7]:
# Settings for the weekly frame. Change them here, nowhere else.
import pandas as pd
DECISION_DATES = ["2026-03-08", "2026-03-15", "2026-03-22", "2026-03-29"]  # weekly, windows do not overlap
MIN_PREV_CLICKS = 10      # page must have at least this many clicks in the previous 7 days
DROP_FRACTION = 0.30      # label = 1 if next-7-day clicks <= (1 - 0.30) * previous-7-day clicks

ROOT = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"
MONTHS = ["2026-02", "2026-03", "2026-04"]   # Feb for the "week before last", Apr for the outcome of late March
FILES = [f"{ROOT}/month={m}/*.parquet" for m in MONTHS]

first = pd.Timestamp(min(DECISION_DATES)) - pd.Timedelta(days=14)
last = pd.Timestamp(max(DECISION_DATES)) + pd.Timedelta(days=7)
date_list = ", ".join(f"DATE '{d}'" for d in DECISION_DATES)

sql = f"""
WITH d AS (
    SELECT UNNEST([{date_list}]) AS decision_date
),
base AS (
    SELECT
        report_date, client_hash_id, content_hash_id,
        gsc_clicks, gsc_impressions,
        NULLIF(gsc_avg_position, 0) AS pos,
        CASE WHEN ga4_data_available IS TRUE THEN sessions_organic END AS organic,
        CASE WHEN ga4_data_available IS TRUE THEN scroll_events END AS scroll,
        (ga4_data_available IS TRUE) AS ga4_ok
    FROM read_parquet({FILES})
    WHERE gsc_data_available IS TRUE          -- zero-filled rows are not real zeros
      AND report_date BETWEEN DATE '{first.date()}' AND DATE '{last.date()}'
),
agg AS (
    SELECT
        d.decision_date, b.client_hash_id, b.content_hash_id,

        COUNT(*)              FILTER (WHERE b.report_date BETWEEN d.decision_date - 7  AND d.decision_date - 1) AS prev_days,
        SUM(b.gsc_clicks)     FILTER (WHERE b.report_date BETWEEN d.decision_date - 7  AND d.decision_date - 1) AS prev_clicks,
        SUM(b.gsc_impressions) FILTER (WHERE b.report_date BETWEEN d.decision_date - 7 AND d.decision_date - 1) AS prev_impr,
        AVG(b.pos)            FILTER (WHERE b.report_date BETWEEN d.decision_date - 7  AND d.decision_date - 1) AS prev_pos,
        COUNT(*)              FILTER (WHERE b.ga4_ok AND b.report_date BETWEEN d.decision_date - 7 AND d.decision_date - 1) AS prev_ga4_days,
        SUM(b.organic)        FILTER (WHERE b.report_date BETWEEN d.decision_date - 7  AND d.decision_date - 1) AS prev_organic,
        SUM(b.scroll)         FILTER (WHERE b.report_date BETWEEN d.decision_date - 7  AND d.decision_date - 1) AS prev_scroll,

        COUNT(*)              FILTER (WHERE b.report_date BETWEEN d.decision_date - 14 AND d.decision_date - 8) AS prior_days,
        SUM(b.gsc_clicks)     FILTER (WHERE b.report_date BETWEEN d.decision_date - 14 AND d.decision_date - 8) AS prior_clicks,

        COUNT(*)              FILTER (WHERE b.report_date BETWEEN d.decision_date + 1  AND d.decision_date + 7) AS future_days,
        SUM(b.gsc_clicks)     FILTER (WHERE b.report_date BETWEEN d.decision_date + 1  AND d.decision_date + 7) AS future_clicks
    FROM d
    JOIN base b ON b.report_date BETWEEN d.decision_date - 14 AND d.decision_date + 7
    GROUP BY d.decision_date, b.client_hash_id, b.content_hash_id
)
SELECT * FROM agg
WHERE prev_days = 7 AND future_days = 7       -- complete windows only
  AND prev_clicks >= {MIN_PREV_CLICKS}
"""

CACHE = Path("w05_weekly_frame.parquet")   # local cache only, never committed (parquet is gitignored)
if CACHE.exists():
    raw = pd.read_parquet(CACHE)
    print("Loaded cached frame:", CACHE)
else:
    raw = con.sql(sql).df()
    raw.to_parquet(CACHE, index=False)
    print("Queried the warehouse and cached the result.")

print("Rows:", f"{len(raw):,}")

Querying Hugging Face warehouse...


HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month=2026-02/data_0.parquet' (HTTP 401)

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.